# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [26]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Add any other imports you need here

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [ ]:
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.
import os
from sklearn.neighbors import NearestNeighbors

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('screenshots', exist_ok=True)

cat_cols = ['city', 'location', 'Status', 'property_type']
encoders = {}
total_unknowns = 0
total_remaining_unknowns = 0

# Fit NearestNeighbors on train coordinates to find nearest locations
train_coords = train_df[['latitude', 'longitude']].values
nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(train_coords)

# Distance threshold (using approx 0.05 degrees, roughly 5-6 km)
DISTANCE_THRESHOLD = 0.05

# Fit LabelEncoder strictly on train to prevent data leakage, and handle unseen labels in test
for col in cat_cols:
    le = LabelEncoder()
    # original train sequence
    orig_train_col_str = train_df[col].astype(str).copy()
    le.fit(orig_train_col_str)
    
    # Append 'Unknown' to classes to handle unseen test labels
    le_classes = list(le.classes_)
    if 'Unknown' not in le_classes:
        le_classes.append('Unknown')
    le.classes_ = np.array(le_classes)
    
    train_df[col] = le.transform(orig_train_col_str)
    
    # transform test data
    test_vals = test_df[col].astype(str)
    known_classes = set(le.classes_)
    
    unseen_mask = ~test_vals.isin(known_classes)
    unknown_count = unseen_mask.sum()
    total_unknowns += unknown_count
    
    if unknown_count > 0:
        unseen_indices = test_df.index[unseen_mask]
        unseen_coords = test_df.loc[unseen_indices, ['latitude', 'longitude']].values
        
        distances, indices = nn.kneighbors(unseen_coords)
        
        # Replace unseen values with nearest neighbor's original value (if within threshold)
        for i, (dist, idx) in enumerate(zip(distances, indices)):
            if dist[0] <= DISTANCE_THRESHOLD:
                # get original train string value using index
                nearest_train_val = orig_train_col_str.iloc[idx[0]]
                test_vals.loc[unseen_indices[i]] = nearest_train_val
            else:
                test_vals.loc[unseen_indices[i]] = 'Unknown'
                
        print(f"[{col}] Replaced {unknown_count} unseen test labels using Nearest Neighbors (Threshold={DISTANCE_THRESHOLD}).")
    
    # Final check just in case, though they should be handled by now
    test_vals_safe = test_vals.apply(lambda x: x if x in known_classes else 'Unknown')
    
    # Calculate how many fell back to 'Unknown' because they were outside threshold
    remaining_unknowns = (test_vals_safe == 'Unknown').sum()
    total_remaining_unknowns += remaining_unknowns
    if unknown_count > 0:
        print(f"[{col}] Remaining 'Unknown' labels after imputation: {remaining_unknowns}\n")

    test_df[col] = le.transform(test_vals_safe)
    
    encoders[col] = le

print(f"Total unseen labels initially: {total_unknowns}")
print(f"Total 'Unknown' labels remaining across all columns: {total_remaining_unknowns}")

with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# TODO: Separate predictors (X) and target (y: 'price')
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']

X_test = test_df.drop('price', axis=1)
y_test = test_df['price']

[location] Replaced 53 unseen test labels using Nearest Neighbors (Threshold=0.05).
[location] Remaining 'Unknown' labels after imputation: 0

Total unseen labels initially: 53
Total 'Unknown' labels remaining across all columns: 0


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [ ]:
rf = RandomForestRegressor(random_state=21)

# TODO: Initialize trackio project/experiment here
# delete any preceding project runs to prevent duplication and clutter
try:
    trackio.delete_project("Rent Prediction Optimization", force=True)
except Exception:
    pass

# TODO: 1. Grid Search Implementation
trackio.init(project="Rent Prediction Optimization", name="Grid_Search")
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 8]
}
gs = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, return_train_score=False)
start_time = time.time()
gs.fit(X_train, y_train)
gs_time = time.time() - start_time
gs_best_score = -gs.best_score_
gs_best_params = gs.best_params_

gs_results = pd.DataFrame(gs.cv_results_)
gs_errors = -gs_results['mean_test_score']
gs_errors_progression = np.minimum.accumulate(gs_errors)

for i in range(len(gs_errors)):
    params = gs_results['params'].iloc[i]
    trackio.log({
        "cv_error": float(gs_errors.iloc[i]),
        "best_cv_error_so_far": float(gs_errors_progression.iloc[i]),
        **params
    }, step=i+1)

trackio.log({"duration_seconds": float(gs_time), "model_version": "RandomForestRegressor_v1"})


# TODO: 2. Random Search Implementation
trackio.init(project="Rent Prediction Optimization", name="Random_Search")
param_dist = {
    'n_estimators': range(50, 201),
    'max_depth': range(10, 31),
    'min_samples_split': range(2, 11)
}
rs = RandomizedSearchCV(rf, param_dist, n_iter=60, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=21, return_train_score=False)
start_time = time.time()
rs.fit(X_train, y_train)
rs_time = time.time() - start_time
rs_best_score = -rs.best_score_
rs_best_params = rs.best_params_

rs_results = pd.DataFrame(rs.cv_results_)
rs_errors = -rs_results['mean_test_score']
rs_errors_progression = np.minimum.accumulate(rs_errors)

for i in range(len(rs_errors)):
    params = rs_results['params'].iloc[i]
    trackio.log({
        "cv_error": float(rs_errors.iloc[i]),
        "best_cv_error_so_far": float(rs_errors_progression.iloc[i]),
        **params
    }, step=i+1)

trackio.log({"duration_seconds": float(rs_time), "model_version": "RandomForestRegressor_v1"})


# TODO: 3. Bayesian Optimization (Optuna) Implementation
trackio.init(project="Rent Prediction Optimization", name="Bayesian_Optimization")
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=21
    )
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1).mean()
    return -score

study = optuna.create_study(direction='minimize')
start_time = time.time()
study.optimize(objective, n_trials=60)
bo_time = time.time() - start_time
bo_best_score = study.best_value
bo_best_params = study.best_params

bo_errors = [trial.value for trial in study.trials]
bo_errors_progression = np.minimum.accumulate(bo_errors)

for i, trial in enumerate(study.trials):
    trackio.log({
        "cv_error": float(trial.value),
        "best_cv_error_so_far": float(bo_errors_progression[i]),
        **trial.params
    }, step=i+1)

trackio.log({"duration_seconds": float(bo_time), "model_version": "RandomForestRegressor_v1"})

* Project 'Rent Prediction Optimization' has been deleted.
* Created new run: Grid_Search


* Created new run: Random_Search


[I 2026-04-10 17:06:22,649] A new study created in memory with name: no-name-5dd387de-ba0c-4e24-8ef1-155372c9dcb5


* Created new run: Bayesian_Optimization


[I 2026-04-10 17:06:24,431] Trial 0 finished with value: 13771.207629256673 and parameters: {'n_estimators': 58, 'max_depth': 19, 'min_samples_split': 6}. Best is trial 0 with value: 13771.207629256673.
[I 2026-04-10 17:06:29,559] Trial 1 finished with value: 14261.014356302732 and parameters: {'n_estimators': 196, 'max_depth': 14, 'min_samples_split': 9}. Best is trial 0 with value: 13771.207629256673.
[I 2026-04-10 17:06:33,943] Trial 2 finished with value: 13918.657438115994 and parameters: {'n_estimators': 142, 'max_depth': 25, 'min_samples_split': 9}. Best is trial 0 with value: 13771.207629256673.
[I 2026-04-10 17:06:37,626] Trial 3 finished with value: 13591.377824926498 and parameters: {'n_estimators': 111, 'max_depth': 27, 'min_samples_split': 5}. Best is trial 3 with value: 13591.377824926498.
[I 2026-04-10 17:06:41,292] Trial 4 finished with value: 13554.701696124208 and parameters: {'n_estimators': 119, 'max_depth': 16, 'min_samples_split': 2}. Best is trial 4 with value: 1

## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [34]:
# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.

plt.figure(figsize=(10, 6))
plt.plot(range(1, 61), gs_errors_progression, label='Grid Search')
plt.plot(range(1, 61), rs_errors_progression, label='Random Search')
plt.plot(range(1, 61), bo_errors_progression, label='Bayesian Optimization (Optuna)')
plt.xlabel('Number of Iterations')
plt.ylabel('Best Mean CV Error so far (MAE)')
plt.title('Compute Budget (Iterations) vs Optimization Error')
plt.legend()
plt.grid(True)
plt.savefig('plots/trials_vs_error.png', bbox_inches='tight')
plt.close()

# TODO: Generate and save optuna_hyperparameter_space.png
try:
    fig = optuna.visualization.plot_contour(study)
    fig.write_image("plots/optuna_hyperparameter_space.png")
    
    fig2 = optuna.visualization.plot_slice(study)
    fig2.write_image("plots/optuna_hyperparameter_space2.png")
except Exception:
    import optuna.visualization.matplotlib as ovm
    fig = ovm.plot_contour(study)
    plt.savefig('plots/optuna_hyperparameter_space.png', bbox_inches='tight')
    plt.close()
    
    fig2 = ovm.plot_slice(study)
    plt.savefig('plots/optuna_hyperparameter_space2.png', bbox_inches='tight')
    plt.close()

/tmp/ipykernel_2343/765055382.py:27: ExperimentalWarning: optuna.visualization.matplotlib._contour.plot_contour is experimental (supported from v2.2.0). The interface can change in the future.
  fig = ovm.plot_contour(study)
/tmp/ipykernel_2343/765055382.py:31: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  fig2 = ovm.plot_slice(study)


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [35]:
# TODO: Print the best hyperparameters found by all 3 methods
print("Grid Search Best Params:", gs_best_params)
print("Random Search Best Params:", rs_best_params)
print("Bayesian Optimization Best Params:", bo_best_params)

# TODO: Train the best model found on the full X_train
best_overall_score = min(gs_best_score, rs_best_score, bo_best_score)
if best_overall_score == gs_best_score:
    best_params = gs_best_params
elif best_overall_score == rs_best_score:
    best_params = rs_best_params
else:
    best_params = bo_best_params

print("Overall Best Params Used:", best_params)

best_model = RandomForestRegressor(**best_params, random_state=21)
best_model.fit(X_train, y_train)

# TODO: Evaluate the model on X_test (Report MAE)
y_pred = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
print(f"Final Test MAE: {test_mae}")

# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
with open('models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

Grid Search Best Params: {'max_depth': 30, 'min_samples_split': 2, 'n_estimators': 200}
Random Search Best Params: {'n_estimators': 158, 'min_samples_split': 2, 'max_depth': 28}
Bayesian Optimization Best Params: {'n_estimators': 127, 'max_depth': 24, 'min_samples_split': 2}
Overall Best Params Used: {'n_estimators': 127, 'max_depth': 24, 'min_samples_split': 2}
Final Test MAE: 12464.453532334706
